# E15 - Playground visual de self-attention y QKV


## Imagen guia del tema <!-- data-aem4-visual-assets -->

![e15 attention qkv playground](assets/visuals/e15_attention_qkv_playground.svg)

Observa esta imagen antes de ejecutar las celdas. Te ayuda a ver el flujo completo antes de mirar numeros.


## Para que sirve

Este laboratorio permite cambiar una oracion y un token objetivo para comprobar que self-attention no mira todo con la misma fuerza. Usa reglas simples por dominio para generar un mapa de atencion didactico.

**Cuando usar este notebook:** despues de `E04_self_attention_con_banco_y_contexto.ipynb` y `E05_query_key_value_como_buscador.ipynb`.

**Idea clave:** attention es una forma de preguntar: para entender este token, que otros tokens me sirven


In [ ]:
# Edita estas entradas para probar nuevos casos.
oraciones = [
    "El banco aprobo el credito del cliente",
    "Me sente en el banco de la plaza",
    "El contrato exige confidencialidad postcontractual",
    "AML thresholding afecta el monitoreo financiero",
]

token_objetivo = "banco"  # probar: banco, contrato, AML, confidencialidad


In [ ]:
import re

dominios = {
    "financiero": {"banco", "credito", "cliente", "AML", "thresholding", "financiero", "monitoreo", "fee", "capping"},
    "plaza": {"banco", "sente", "plaza", "madera", "asiento"},
    "legal": {"contrato", "confidencialidad", "postcontractual", "clausula", "obligacion"},
}

def tokens(texto):
    return re.findall(r"[A-Za-z]+", texto)

def detectar_dominio(palabras):
    scores = {}
    low = {p.lower() for p in palabras}
    for nombre, vocab in dominios.items():
        scores[nombre] = len(low & {v.lower() for v in vocab})
    return max(scores, key=scores.get), scores

def attention_didactica(oracion, objetivo):
    toks = tokens(oracion)
    dominio, scores = detectar_dominio(toks)
    vocab = {v.lower() for v in dominios[dominio]}
    pesos = []
    for t in toks:
        peso = 0.05
        if t.lower() == objetivo.lower():
            peso += 0.20
        if t.lower() in vocab:
            peso += 0.25
        if len(t) > 10:
            peso += 0.10
        pesos.append((t, round(min(peso, 0.90), 2)))
    total = sum(p for _, p in pesos) or 1
    normalizados = [(t, round(p / total, 2)) for t, p in pesos]
    return dominio, scores, normalizados


In [ ]:
for oracion in oraciones:
    dominio, scores, pesos = attention_didactica(oracion, token_objetivo)
    print("\nORACION:", oracion)
    print("Dominio detectado:", dominio, "| scores:", scores)
    print("Mapa de atencion didactico para:", token_objetivo)
    for token, peso in sorted(pesos, key=lambda x: x[1], reverse=True):
        print(f"  {token:>18} | {peso:.2f} {'#' * int(peso * 40)}")


In [ ]:
# Q/K/V como buscador: Query del token objetivo contra Keys de cada token.
def query_para(objetivo):
    objetivo = objetivo.lower()
    if objetivo in {"banco", "credito", "aml"}:
        return {"financiero", "credito", "cliente", "monitoreo"}
    if objetivo in {"contrato", "confidencialidad"}:
        return {"legal", "obligacion", "clausula", "riesgo"}
    return {"contexto", "significado"}

for oracion in oraciones:
    q = query_para(token_objetivo)
    print("\nQuery conceptual:", q)
    for t in tokens(oracion):
        _, _, pesos = attention_didactica(t, token_objetivo)
        key = set()
        for dominio, vocab in dominios.items():
            if t.lower() in {v.lower() for v in vocab}:
                key.add(dominio)
        match = len(q & key)
        print(f"{t:>18} | Key={key or {'general'}} | match={match}")


## Preguntas para responder

1. Que tokens reciben mas peso y por que
2. Cambia el mapa si `banco` aparece con `credito` o con `plaza`
3. Que seria la Query del token objetivo
4. Que ofrece cada Key

## Mini desafio

Cambiar `token_objetivo` y agregar una oracion ambigua. Predice el mapa antes de ejecutar y despues compara tu intuicion con la salida.
